In [3]:
import pandas as pd
import kagglehub
import geopandas as gpd
import datetime as dt

# Download latest version
path = kagglehub.dataset_download("alexgude/california-traffic-collision-data-from-switrs")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\madel\.cache\kagglehub\datasets\alexgude\california-traffic-collision-data-from-switrs\versions\6


## Data Cleaning

### Grab data for synethic control

In [4]:
import sqlite3
import os

# The 'path' variable is already defined from the kagglehub.dataset_download call
# in the first cell, which correctly points to the dataset's directory.
# We should use that path instead of a hardcoded incorrect one.
db_file_path = os.path.join(path, 'switrs.sqlite') # Changed path_to_dataset_dir to path
conn = sqlite3.connect(db_file_path)

collisions = pd.read_sql_query("""
    SELECT case_id, collision_date, county_city_location, collision_severity, county_location, population, weather_1, primary_collision_factor, pcf_violation_category, hit_and_run, type_of_collision, road_condition_1, road_surface, lighting, alcohol_involved, latitude, longitude
    FROM collisions
    WHERE collision_date >= '2010-01-01' AND collision_date <= '2016-12-31';
""", conn)

collisions.head()

,case_id,collision_date,county_city_location,collision_severity,county_location,population,weather_1,primary_collision_factor,pcf_violation_category,hit_and_run,type_of_collision,road_condition_1,road_surface,lighting,alcohol_involved,latitude,longitude
0,4391974,2010-01-06,1942,property damage only,los angeles,>250000,clear,other than driver,other than driver (or pedestrian),not hit and run,hit object,normal,dry,daylight,NaN,NaN,NaN
1,4392005,2010-01-19,3300,fatal,riverside,unincorporated,cloudy,vehicle code violation,improper turning,not hit and run,hit object,normal,dry,dark with street lights,1.0,34.01847,-117.50958
2,4392006,2010-01-13,1935,fatal,los angeles,50000 to 100000,raining,vehicle code violation,traffic signals and signs,not hit and run,broadside,normal,wet,dark with street lights,NaN,NaN,NaN
3,4392007,2010-01-15,1942,fatal,los angeles,>250000,clear,vehicle code violation,pedestrian violation,felony,pedestrian,normal,dry,dark with street lights,1.0,NaN,NaN
4,4392008,2010-01-03,1942,fatal,los angeles,>250000,cloudy,vehicle code violation,traffic signals and signs,not hit and run,broadside,normal,dry,daylight,NaN,NaN,NaN


In [5]:
print(collisions.shape)
print(collisions.columns)

(2929213, 17)
Index(['case_id', 'collision_date', 'county_city_location',
       'collision_severity', 'county_location', 'population', 'weather_1',
       'primary_collision_factor', 'pcf_violation_category', 'hit_and_run',
       'type_of_collision', 'road_condition_1', 'road_surface', 'lighting',
       'alcohol_involved', 'latitude', 'longitude'],
      dtype='object')


### Organize Cities by NCIC Codes

In [6]:
NCIC = pd.read_csv('NCIC Code Jurisdiction List_04242023 - Sheet1.csv')

In [7]:
#combine data sets
collisions_coded = pd.merge(collisions, NCIC, how='left', left_on='county_city_location', right_on='Code')

In [8]:
#get value counts of codes
print(collisions_coded['Code'].value_counts())

Code
1942    354735
1900    107641
3711     63027
3400     58781
0109     47334
         ...  
1010         1
1504         1
1307         1
1303         1
1204         1
Name: count, Length: 498, dtype: int64


In [9]:
#drop codes for Los Angles, San Diego, San Jose, and Fremont
collisions_ca = collisions_coded[~collisions_coded['Code'].isin(['1942', '3711', '4313', '0105'])]

In [10]:
print(collisions_ca['Code'].value_counts())

Code
1900    107641
3400     58781
0109     47334
1941     45573
3801     44833
         ...  
4707         1
4710         1
1307         1
1303         1
1204         1
Name: count, Length: 494, dtype: int64


### Label treatment by city and time


In [11]:
collisions_ca.columns

Index(['case_id', 'collision_date', 'county_city_location',
       'collision_severity', 'county_location', 'population', 'weather_1',
       'primary_collision_factor', 'pcf_violation_category', 'hit_and_run',
       'type_of_collision', 'road_condition_1', 'road_surface', 'lighting',
       'alcohol_involved', 'latitude', 'longitude', 'CntyCode', 'County',
       'Code', 'Agency', 'Start', 'End'],
      dtype='object')

In [12]:
collisions_ca['collision_date'] = pd.to_datetime(collisions_ca['collision_date'])
collisions_ca['year_month'] = collisions_ca['collision_date'].dt.to_period('M')

C:\Users\madel\AppData\Local\Temp\ipykernel_1912\3713592967.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  collisions_ca['collision_date'] = pd.to_datetime(collisions_ca['collision_date'])
C:\Users\madel\AppData\Local\Temp\ipykernel_1912\3713592967.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  collisions_ca['year_month'] = collisions_ca['collision_date'].dt.to_period('M')


In [13]:
print(collisions_ca.shape)
print(collisions_ca.columns)

(2453441, 24)
Index(['case_id', 'collision_date', 'county_city_location',
       'collision_severity', 'county_location', 'population', 'weather_1',
       'primary_collision_factor', 'pcf_violation_category', 'hit_and_run',
       'type_of_collision', 'road_condition_1', 'road_surface', 'lighting',
       'alcohol_involved', 'latitude', 'longitude', 'CntyCode', 'County',
       'Code', 'Agency', 'Start', 'End', 'year_month'],
      dtype='object')


In [14]:
#Get rid of Sheriff's Departments in Agencys
collisions_ca = collisions_ca[~collisions_ca['Agency'].str.contains('Sheriff', na=False)]
collisions_ca = collisions_ca[~collisions_ca['Agency'].str.contains('Police', na=False)]

print(collisions_ca['Agency'].unique())

['Lakewood' 'Rancho Cucamonga' 'Fresno' 'South El Monte' 'Barstow'
 'Pomona' 'Walnut' 'Hanford' 'Los Gatos' nan 'Ontario' 'Brisbane'
 'Huntington Beach' 'Fullerton' 'Tehachapi' 'Compton' 'Costa Mesa'
 'Sacramento' 'Long Beach' 'Corona' 'Roseville' 'La Canada-Flintridge'
 'San Francisco' 'Palo Alto' 'Turlock' 'El Cajon' 'Hemet' 'Redding'
 'Westminster' 'Fontana' 'Palmdale' 'Orange' 'Bakersfield' 'Eureka'
 'West Covina' 'Bellflower' 'Industry' 'Modesto' 'Riverside' 'Bell'
 'Palm Springs' 'Seal Beach' 'Irvine' 'Huntington Park' 'Yuba City'
 'Emeryville' 'Downey' 'Rialto' 'Vista' 'Fairfield' 'Carlsbad' 'Chico'
 'Manteca' 'Ripon' 'Covina' 'Temple City' 'Fountain Valley' 'Fortuna'
 'Inglewood' 'Fillmore' 'Stockton' 'Gilroy' 'Needles' 'Antioch'
 'National City' 'Commerce' 'Twentynine Palms' 'Ventura' 'Escondido'
 'Oakland' 'Richmond' 'Port Hueneme' 'Pittsburg' 'Banning' 'Santa Ana'
 'Santa Fe Springs' 'Rocklin' 'Tulare' 'Concord' 'Cerritos' 'Glendale'
 'Lompoc' 'La Palma' 'Hawthorne' 'Buena P

In [15]:
#Find top 18 cities plus San Francisco in terms of crashes by Agency Name
top_cities = collisions_ca.groupby('Agency')['case_id'].count().sort_values(ascending = False)
top_cities = top_cities.head(19)

In [16]:
collisions_ca.shape

(1779571, 24)

In [17]:
collisions_ca.to_csv('California_Collisions_Clean.csv')

KeyboardInterrupt: 